# Installing important libraries

In [24]:
!pip install  openai langchain chromadb faiss-cpu pypdf tiktoken docarray PyPDF tiktoken


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
!pip install langchain-openai


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [140]:
%pip install -qU  flashrank

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
!pip install -qU langchain-community pypdf pillow


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
from dotenv import load_dotenv
import os
import shutil
import time
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma, FAISS 


In [28]:
!pip install -qU "langchain-chroma>=0.1.2"


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
load_dotenv()

True

In [30]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [67]:


llm = ChatOpenAI(model_name="gpt-4o-mini"
                    ,streaming=True)

In [32]:
# Load PDF documents with error handling
print("Loading PDF documents from ./Policy+Documents...")
try:
    pdf_directory_loader = PyPDFDirectoryLoader("./Policy+Documents")
    documents = pdf_directory_loader.load()
    print(f"✓ Successfully loaded {len(documents)} documents")
    print(f"✓ Total pages: {sum(doc.metadata.get('total_pages', 1) for doc in documents)}")
except Exception as e:
    print(f"Error loading documents: {str(e)}")
    raise

Loading PDF documents from ./Policy+Documents...
✓ Successfully loaded 217 documents
✓ Total pages: 7209


In [33]:
documents[0].page_content[:100]

'Part A \n<<Date>> \n<<Policyholder’s Name>>  \n<<Policyholder’s Address>> \n<<Policyholder’s Contact Num'

In [34]:
# Split documents into chunks
print("Splitting documents into chunks...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)
splits = text_splitter.split_documents(documents)
print(f"✓ Created {len(splits)} document chunks")
print(f"✓ Average chunk size: {sum(len(s.page_content) for s in splits) // len(splits)} characters")

Splitting documents into chunks...
✓ Created 760 document chunks
✓ Average chunk size: 848 characters


In [35]:
print(splits[0])

page_content='Part A 
<<Date>> 
<<Policyholder’s Name>>  
<<Policyholder’s Address>> 
<<Policyholder’s Contact Number>> 
 
Dear <<Policyholder’s Name>>,  
 
Sub: Your Policy no. <<  >> 
We are glad to inform you that your proposal has been accepted and the HDFC Life Easy Health (“Policy”) 
being this document, has been issued. We have made every effort to design your Policy in a simple format. We 
have highlighted items of importance so that you may recognize them easily. 
 
Policy document: 
As an evidence of the insurance contract between HDFC Life Insurance Company Limited and you, the Policy 
is enclosed herewith. Please preserve this document safely and also inform your nominees about the same. A 
copy of your proposal form and other relevant documents submitted by you is also enclosed for your 
information and record.  
 
Cancellation in the Free-Look Period: 
 
<< In case you are not agreeable to any of the terms and conditions stated in the Policy, you have the option to' metad

In [36]:
# Initialize embeddings model with timeout and retry settings
print("Initializing OpenAI embeddings model...")
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",  # Using smaller, faster model
    request_timeout=60,  # 60 second timeout
    max_retries=3  # Retry up to 3 times on failure
)
print("✓ Embeddings model initialized")

Initializing OpenAI embeddings model...
✓ Embeddings model initialized


In [37]:
# Test embedding on a single document
print("Testing embeddings on a sample chunk...")
try:
    test_embedding = embeddings_model.embed_documents([splits[0].page_content])
    print(f"✓ Test embedding successful - dimension: {len(test_embedding[0])}")
except Exception as e:
    print(f"✗ Error testing embeddings: {str(e)}")
    raise

Testing embeddings on a sample chunk...
✓ Test embedding successful - dimension: 1536


In [38]:
# Check test embedding dimensions (already done in cell 16)
# The test_embedding variable was created in cell 16
# Uncomment below if you want to check again:
# len(test_embedding), len(test_embedding[0])
print("Embedding test completed in cell 16")

Embedding test completed in cell 16


In [39]:
# Check embedding type (already done in cell 16)
# The test_embedding variable was created in cell 16
# Uncomment below if you want to check again:
# type(test_embedding)
print("Embedding type: list (as shown in cell 16)")

Embedding type: list (as shown in cell 16)


In [40]:
# store = LocalFileStore("./cache/") 

# cached_embedder = CacheBackedEmbeddings.from_bytes_store(
#     embeddings_model,
#     store,
#     namespace="semantic-spotter"
# )

In [41]:
# Clear existing chroma_store if it exists to avoid conflicts
if os.path.exists("./chroma_store"):
    print("Removing existing chroma_store directory...")
    try:
        shutil.rmtree("./chroma_store")
        print("✓ Existing chroma_store removed successfully!")
    except Exception as e:
        print(f"Warning: Could not remove existing directory: {e}")

print(f"\n{'='*60}")
print(f"Total document splits to process: {len(splits)}")
print(f"Estimated processing time: {len(splits) * 0.1:.1f} seconds (~{len(splits) * 0.1 / 60:.1f} minutes)")
print(f"{'='*60}\n")



Total document splits to process: 760
Estimated processing time: 76.0 seconds (~1.3 minutes)



In [42]:
# Preview first few splits (safe check)
print("Preview of document splits:")
try:
    for i, split in enumerate(splits[:3]):
        print(f"\n--- Split {i+1} ---")
        print(f"Source: {split.metadata.get('source', 'Unknown')}")
        print(f"Page: {split.metadata.get('page', 'Unknown')}")
        print(f"Content preview: {split.page_content[:150]}...")
    print(f"\n✓ Total splits available: {len(splits)}")
except Exception as e:
    print(f"Error previewing splits: {e}")
    raise

Preview of document splits:

--- Split 1 ---
Source: Policy+Documents\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Page: 0
Content preview: Part A 
<<Date>> 
<<Policyholder’s Name>>  
<<Policyholder’s Address>> 
<<Policyholder’s Contact Number>> 
 
Dear <<Policyholder’s Name>>,  
 
Sub: Yo...

--- Split 2 ---
Source: Policy+Documents\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Page: 0
Content preview: information and record.  
 
Cancellation in the Free-Look Period: 
 
<< In case you are not agreeable to any of the terms and conditions stated in the...

--- Split 3 ---
Source: Policy+Documents\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Page: 0
Content preview: option to return the Policy to us for cancellation stating the reasons thereof, within 15 days from the date of 
receipt of the Policy. On receipt of ...

✓ Total splits available: 760


In [ ]:
# SAFE APPROACH: Use FAISS first (in-memory, more stable) then optionally save to Chroma
# This avoids ChromaDB initialization issues on Windows
import gc
import pickle

def create_vector_store_faiss(splits, embeddings_model, save_path="./faiss_store"):
    """
    Create vector store using FAISS (in-memory, more stable than ChromaDB on Windows).
    This should not crash the kernel.
    """
    print(f"{'='*60}")
    print("Creating vector store using FAISS (in-memory)...")
    print(f"Total document chunks: {len(splits)}")
    print(f"{'='*60}\n")
    
    start_time = time.time()
    
    try:
        # Step 1: Test embeddings first
        print("Step 1: Testing embeddings on first document...")
        try:
            test_embedding = embeddings_model.embed_documents([splits[0].page_content])
            print(f"✓ Test embedding successful (dimension: {len(test_embedding[0])})")
            gc.collect()
        except Exception as e:
            print(f"✗ Error testing embeddings: {e}")
            raise
        
        # Step 2: Create FAISS vector store (this is in-memory, should be safe)
        print("\nStep 2: Creating FAISS vector store...")
        print("Processing documents in small batches to prevent memory issues...\n")
        
        # Process in small batches to avoid memory issues
        batch_size = 10
        all_docs = []
        
        for i in range(0, len(splits), batch_size):
            batch = splits[i:i+batch_size]
            batch_num = (i // batch_size) + 1
            total_batches = (len(splits) + batch_size - 1) // batch_size
            
            print(f"Processing batch {batch_num}/{total_batches} ({len(batch)} documents)...", end=" ", flush=True)
            
            try:
                # Embed the batch
                texts = [doc.page_content for doc in batch]
                embeddings = embeddings_model.embed_documents(texts)
                
                # Store for FAISS creation
                all_docs.extend(batch)
                
                print("✓")
                
                # Small delay and garbage collection
                if batch_num < total_batches:
                    gc.collect()
                    time.sleep(0.5)
                    
            except Exception as e:
                print(f"✗ Error in batch {batch_num}: {e}")
                print("Continuing with next batch...")
                continue
        
        # Step 3: Create FAISS vector store from all documents
        print(f"\nStep 3: Creating FAISS index from {len(all_docs)} documents...")
        try:
            vectordb = FAISS.from_documents(
                documents=all_docs,
                embedding=embeddings_model
            )
            print("✓ FAISS vector store created successfully")
        except Exception as e:
            print(f"✗ Error creating FAISS store: {e}")
            raise
        
        # Step 4: Save to disk (PERSISTENT STORAGE)
        print(f"\nStep 4: Saving vector store to disk (persistent storage)...")
        try:
            os.makedirs(os.path.dirname(save_path) if os.path.dirname(save_path) else ".", exist_ok=True)
            vectordb.save_local(save_path)
            print(f"✓ Vector store saved to: {save_path}")
            print(f"✓ Vector store is PERSISTENT - it will survive kernel restarts!")
            print(f"✓ To load it later, use: load_saved_vector_store('{save_path}')")
            
            # Verify the save
            if os.path.exists(save_path):
                file_count = len(os.listdir(save_path))
                print(f"✓ Verification: {file_count} files saved to disk")
        except Exception as e:
            print(f"⚠ Warning: Could not save to disk: {e}")
            print("Vector store is in memory only (will be lost when kernel restarts)")
        
        elapsed_time = time.time() - start_time
        print(f"\n{'='*60}")
        print(f"✓ Successfully created FAISS vector store!")
        print(f"✓ Total documents processed: {len(all_docs)}")
        print(f"✓ Time taken: {elapsed_time:.1f} seconds ({elapsed_time/60:.1f} minutes)")
        print(f"{'='*60}\n")
        
        return vectordb
        
    except Exception as e:
        print(f"\n{'='*60}")
        print(f"✗ Error: {e}")
        print(f"Error type: {type(e).__name__}")
        print(f"{'='*60}")
        raise

# Try FAISS approach (more stable on Windows)
print("Creating vector store using FAISS (in-memory approach)...")
print("This should be more stable and prevent kernel crashes.\n")

try:
    vectordb = create_vector_store_faiss(splits, embeddings_model, "./faiss_store")
    print("✓ Vector store creation completed successfully!")
    print("\nNote: FAISS vector store is ready. You can use it for similarity search.")
except Exception as e:
    print(f"\n{'='*60}")
    print("ERROR: FAISS method also failed")
    print(f"Error: {str(e)}")
    print(f"Error type: {type(e).__name__}")
    print(f"{'='*60}")
    print("\nThis suggests the issue might be with:")
    print("1. OpenAI API connection/authentication")
    print("2. Memory issues on your system")
    print("3. Python environment issues")
    print("\nTry:")
    print("1. Restart the kernel completely")
    print("2. Check OpenAI API key: print(os.getenv('OPENAI_API_KEY')[:10] + '...')")
    print("3. Test embeddings manually: embeddings_model.embed_documents(['test'])")
    raise

Creating vector store using FAISS (in-memory approach)...
This should be more stable and prevent kernel crashes.

Creating vector store using FAISS (in-memory)...
Total document chunks: 760

Step 1: Testing embeddings on first document...
✓ Test embedding successful (dimension: 1536)

Step 2: Creating FAISS vector store...
Processing documents in small batches to prevent memory issues...

Processing batch 1/76 (10 documents)... ✓
Processing batch 2/76 (10 documents)... ✓
Processing batch 3/76 (10 documents)... ✓
Processing batch 4/76 (10 documents)... ✓
Processing batch 5/76 (10 documents)... ✓
Processing batch 6/76 (10 documents)... ✓
Processing batch 7/76 (10 documents)... ✓
Processing batch 8/76 (10 documents)... ✓
Processing batch 9/76 (10 documents)... ✓
Processing batch 10/76 (10 documents)... ✓
Processing batch 11/76 (10 documents)... ✓
Processing batch 12/76 (10 documents)... ✓
Processing batch 13/76 (10 documents)... ✓
Processing batch 14/76 (10 documents)... ✓
Processing batc

In [ ]:
# Verify that the vector store is properly saved to disk
print("Verifying vector store persistence...")
print(f"{'='*60}")

# Check if the store directory exists
if os.path.exists("./faiss_store"):
    print("✓ FAISS store directory exists on disk")
    
    # List files in the store
    store_files = os.listdir("./faiss_store")
    print(f"✓ Found {len(store_files)} files in ./faiss_store")
    for file in store_files[:5]:  # Show first 5 files
        file_path = os.path.join("./faiss_store", file)
        file_size = os.path.getsize(file_path) / (1024 * 1024)  # Size in MB
        print(f"  - {file}: {file_size:.2f} MB")
    if len(store_files) > 5:
        print(f"  ... and {len(store_files) - 5} more files")
    
    # Try loading it to verify it works
    print("\nTesting load functionality...")
    try:
        test_vectordb = FAISS.load_local(
            folder_path="./faiss_store",
            embeddings=embeddings_model,
            allow_dangerous_deserialization=True
        )
        print("✓ Vector store loaded successfully from disk!")
        
        # Test a quick search to verify it works
        test_results = test_vectordb.similarity_search("test query", k=1)
        print(f"✓ Search test successful - store is fully functional")
        print(f"\n{'='*60}")
        print("✓ Vector store is properly persisted and ready for future use!")
        print(f"{'='*60}\n")
    except Exception as e:
        print(f"✗ Error loading store: {e}")
else:
    print("✗ FAISS store directory not found")
    print("The vector store may not have been saved properly.")


In [ ]:
# Function to load the saved vector store (use this in future sessions)
def load_saved_vector_store(store_path="./faiss_store"):
    """
    Load the saved FAISS vector store from disk.
    Use this function when you restart your kernel or in a new session.
    """
    print(f"Loading vector store from: {store_path}")
    print(f"{'='*60}")
    
    if not os.path.exists(store_path):
        print(f"✗ Vector store not found at: {store_path}")
        print("Please run cell 23 first to create the vector store.")
        return None
    
    try:
        # Load the vector store
        vectordb = FAISS.load_local(
            folder_path=store_path,
            embeddings=embeddings_model,
            allow_dangerous_deserialization=True
        )
        
        print("✓ Vector store loaded successfully!")
        
        # Get some info about the store
        # Note: FAISS doesn't have a direct way to count documents, but we can test it
        print("✓ Vector store is ready for similarity search")
        print(f"{'='*60}\n")
        
        return vectordb
    except Exception as e:
        print(f"✗ Error loading vector store: {str(e)}")
        print(f"Error type: {type(e).__name__}")
        return None

# Example: Load the saved store (uncomment to use)
# vectordb = load_saved_vector_store("./faiss_store")


In [44]:
# Test the vector store with a similarity search
def similarity_search(query, k=4, vectordb=None):
    """
    Perform similarity search on the vector database.
    """
    if vectordb is None:
        # Try to load existing vector store
        try:
            # Try loading FAISS first, then Chroma
            if os.path.exists("./faiss_store"):
                try:
                    vectordb = FAISS.load_local(
                        folder_path="./faiss_store",
                        embeddings=embeddings_model,
                        allow_dangerous_deserialization=True
                    )
                    print("✓ Loaded existing FAISS vector store")
                except Exception:
                    pass
            
            # If FAISS didn't work, try Chroma
            if vectordb is None and os.path.exists("./chroma_store"):
                try:
                    if 'embeddings_model' not in globals():
                        print("✗ Embeddings model not found. Please run cells 15-16 first.")
                        return None
                    vectordb = Chroma(
                        persist_directory="./chroma_store",
                        embedding_function=embeddings_model
                    )
                    print("✓ Loaded existing Chroma vector store")
                except Exception:
                    pass
            
            if vectordb is None:
                print("✗ Vector store not found. Please create it first by running cell 23.")
                return None
        except Exception as e:
            print(f"✗ Error loading vector store: {str(e)}")
            print(f"Error type: {type(e).__name__}")
            return None
    
    if vectordb is None:
        print("✗ Cannot perform search - vector store is None")
        return None
    
    try:
        print(f"\nSearching for: '{query}'")
        print(f"Retrieving top {k} results...\n")
        results = vectordb.similarity_search(query, k=k)
        
        if not results or len(results) == 0:
            print("No results found.")
            return []
        
        print(f"Found {len(results)} results:\n")
        for i, result in enumerate(results, 1):
            print(f"{'='*60}")
            print(f"Result {i}:")
            print(f"Source: {result.metadata.get('source', 'Unknown')}")
            print(f"Page: {result.metadata.get('page', 'Unknown')}")
            print(f"\nContent preview:")
            content = result.page_content
            if len(content) > 300:
                print(content[:300] + "...")
            else:
                print(content)
            print()
        
        return results
    except Exception as e:
        print(f"✗ Error during search: {str(e)}")
        print(f"Error type: {type(e).__name__}")
        return None

# Test the search functionality (only if vectordb was created successfully)
if 'vectordb' in globals() and vectordb is not None:
    test_query = "What is the life insurance policy coverage amount?"
    similarity_search(test_query, k=4, vectordb=vectordb)
else:
    print("⚠ Vector store not available. Please run cell 23 first to create the vector store.")


Searching for: 'What is the life insurance policy coverage amount?'
Retrieving top 4 results...

Found 4 results:

Result 1:
Source: Policy+Documents\HDFC-Life-Sanchay-Plus-Life-Long-Income-Option-101N134V19-Policy-Document.pdf
Page: 7

Content preview:
on Death which is higher of: 
a) 10 times the Annualized Premium or 1.25 times the Single Premium, or 
b) 105% of Total Premiums paid, or 
c) Premiums paid accumulated at an interest of 5% p.a. compounded annually, or 
d) Guaranteed Sum Assured on Maturity, or 
e) an absolute amount assured to be pa...

Result 2:
Source: Policy+Documents\HDFC-Life-Sanchay-Plus-Life-Long-Income-Option-101N134V19-Policy-Document.pdf
Page: 16

Content preview:
HDFC Life Sanchay Plus (UIN – 101N134V19) – Appendix 9 (c) – Policy Bond 
A non-participating, non-linked savings insurance plan 
 
 
 Page 17 of 27 
 
13                 100% 90% 90% 90% 90% 90% 90% 90% 
14                   100% 90% 90% 90% 90% 90% 90% 
15                     100% 90% 90% 90% 90% 

In [45]:
# Function to load existing vector store (supports both FAISS and Chroma)
def load_vector_store(store_path="./faiss_store", store_type="faiss"):
    """
    Load an existing vector store from disk.
    Supports both FAISS and Chroma stores.
    """
    try:
        if store_type.lower() == "faiss":
            if not os.path.exists(store_path):
                print(f"✗ FAISS vector store not found at: {store_path}")
                return None
            
            vectordb = FAISS.load_local(
                folder_path=store_path,
                embeddings=embeddings_model,
                allow_dangerous_deserialization=True
            )
            print(f"✓ Successfully loaded FAISS vector store from: {store_path}")
            return vectordb
        else:  # Chroma
            if not os.path.exists(store_path):
                print(f"✗ Chroma vector store not found at: {store_path}")
                return None
            
            vectordb = Chroma(
                persist_directory=store_path,
                embedding_function=embeddings_model
            )
            print(f"✓ Successfully loaded Chroma vector store from: {store_path}")
            return vectordb
    except Exception as e:
        print(f"✗ Error loading vector store: {str(e)}")
        return None

# Uncomment to load existing vector store:
# vectordb = load_vector_store("./faiss_store", "faiss")


In [ ]:
# Install required packages for advanced retrieval
%pip install -q langchain-community sentence-transformers


In [46]:
# Advanced search with similarity scores
def similarity_search_with_score(query, k=4, vectordb=None):
    """
    Perform similarity search with relevance scores.
    """
    if vectordb is None:
        vectordb = load_vector_store()
        if vectordb is None:
            return None
    
    try:
        print(f"\nSearching for: '{query}'")
        print(f"Retrieving top {k} results with scores...\n")
        results = vectordb.similarity_search_with_score(query, k=k)
        
        print(f"Found {len(results)} results:\n")
        for i, (result, score) in enumerate(results, 1):
            print(f"{'='*60}")
            print(f"Result {i} (Score: {score:.4f}):")
            print(f"Source: {result.metadata.get('source', 'Unknown')}")
            print(f"Page: {result.metadata.get('page', 'Unknown')}")
            print(f"\nContent preview:")
            print(result.page_content[:300] + "..." if len(result.page_content) > 300 else result.page_content)
            print()
        
        return results
    except Exception as e:
        print(f"✗ Error during search: {str(e)}")
        return None

# Example usage:
# similarity_search_with_score("What are the policy terms and conditions?", k=3, vectordb=vectordb)


In [54]:
vectordb = load_vector_store()

✓ Successfully loaded FAISS vector store from: ./faiss_store


In [55]:
vectordb.as_retriever()

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002CEFCD37B10>, search_kwargs={})

In [ ]:
from langchain.tools import tool
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank

compressor = FlashrankRerank()
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=vectordb.as_retriever(search_kwargs={"k": 20})
)

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query"""
    retrieved_docs = compression_retriever.invoke(
    query
)
    # retrieved_docs = vectordb.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\n Page Content: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [136]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

tools = [retrieve_context]

prompt = """You are an insurance policy QA assistant. Answer questions ONLY based on the policy documents provided in details.

CRITICAL OUTPUT FORMAT RULES:
1. ALWAYS respond with EXACTLY this format, nothing else:
answer: [your answer here]
source: [metadata info from retrieved document pdf name and page no.]
2. Do NOT include any explanation, preamble, or extra text
3. Do NOT repeat the question
4. Do NOT include metadata (producer, creator, page, author, etc.)
5. Use only the actual policy content
6. If answer not found, write: "Not found in the provided policy context."

EXAMPLE OUTPUT:
answer: The policy covers up to $500,000 for life insurance
"""

prompt = """You are an insurance policy QA assistant. Answer questions ONLY based on the policy documents provided in details.

CRITICAL OUTPUT FORMAT RULES:
1. ALWAYS respond with EXACTLY this format, nothing else:
answer: [your answer here]
source: [metadata info from retrieved document pdf name and page no.]

EXAMPLE OUTPUT:
answer: The policy covers up to $500,000 for life insurance
"""

agent = create_agent(llm, tools, system_prompt=prompt)


In [ ]:
query = "What is the life insurance policy coverage amount?"

for event in agent.stream({
    "messages": [{"role": "user", "content": query}]
}, stream_mode="values"):
    print(event["messages"][-1].content)


What is the life insurance policy coverage amount?

("Source: {'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2023-08-24T14:00:00+05:30', 'author': 'harinis', 'moddate': '2023-08-24T14:00:00+05:30', 'source': 'Policy+Documents\\\\HDFC-Life-Sanchay-Plus-Life-Long-Income-Option-101N134V19-Policy-Document.pdf', 'total_pages': 27, 'page': 16, 'page_label': '17'}\n Page Content: HDFC Life Sanchay Plus (UIN – 101N134V19) – Appendix 9 (c) – Policy Bond \nA non-participating, non-linked savings insurance plan \n \n \n Page 17 of 27 \n \n13                 100% 90% 90% 90% 90% 90% 90% 90% \n14                   100% 90% 90% 90% 90% 90% 90% \n15                     100% 90% 90% 90% 90% 90% \n16                       100% 90% 90% 90% 90% \n17                         100% 90% 90% 90% \n18                           100% 100% 100% \n19                             100% 100% \n20                               100% \n \nAppendix 2: Death Benefit Multiple \nSum A

In [137]:
from langchain.messages import HumanMessage

def insurance_agent(query: str):
    response = agent.invoke({
        'messages': [
            HumanMessage(content=(
            query
            ))
        ]
    })

    print(response['messages'][-1].content)

In [121]:
insurance_agent( "What is the life insurance policy coverage amount?")

answer: Sum Assured will be determined based on your entry age and the Annualized Premium you commit to pay in a policy year.
source: Policy+Documents\HDFC-Life-Sanchay-Plus-Life-Long-Income-Option-101N134V19-Policy-Document.pdf, page 16


In [139]:
insurance_agent( "Can a 100 year plus person do a term insurance?")

answer: Generally, term insurance is not available for individuals above a certain age, often around 60-70 years, but specific eligibility may vary by policy and insurer. 
source: HDFC-Life-Sanchay-Plus-Life-Long-Income-Option-101N134V19-Policy-Document.pdf, page 12.


In [134]:
insurance_agent("what is the Definitions of Critical Illnesses? based on policy?")

answer: Definitions of Critical Illnesses include: Myocardial Infarction (First Heart Attack of specific severity), which is defined as the first occurrence of heart attack resulting in the death of a portion of the heart muscle due to inadequate blood supply, evidenced by clinical symptoms, electrocardiogram changes, and elevation of specific enzymes. Exclusions include other acute Coronary Syndromes and certain cardiac biomarker elevations without overt ischemic heart disease.
source: HDFC Life Group Poorna Suraksha (101N137V02) - Policy Document, Page 27


In [126]:
retrieve_context("what is the Definitions of Critical Illnesses? based on policy?")

("Source: {'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2021-11-29T10:03:02+00:00', 'author': 'ANINDYAA', 'moddate': '2021-12-09T06:23:28+00:00', 'title': 'HDFC Life Easy Health - 101N110V03 - Policy Bond (Single Pay)', 'source': 'Policy+Documents\\\\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf', 'total_pages': 33, 'page': 10, 'page_label': '11'}\n Page Content: Part B of this Policy and the Policy is in force on the date of the diagnosis..  \nii. If the diagnosis of the Critical Illness is made within the Policy Term and the 30 days survival period \ncrosses the Policy Term, a valid claim arising as a result of such a diagnosis within the Policy Term \nshall not be denied. \niii. Critical Illness Benefit will be payable only once during the Policy Term. \niv. A waiting period of 90 days as mentioned under Part F (Clause 1) is applicable for availing the Critical \nIllness Benefit failing which we will not pay any benefit to the Life Assure

In [128]:
retrieve_context("Can a 100 year plus person do a term insurance?")

("Source: {'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': 'D:20230821052617', 'title': 'Exide Life Income Advantage Plan – Terms and Conditions', 'author': 'mayurc', 'moddate': 'D:20230821052617', 'source': 'Policy+Documents\\\\HDFC-Life-Sampoorna-Jeevan-101N158V04-Policy-Document (1).pdf', 'total_pages': 44, 'page': 35, 'page_label': '36'}\n Page Content: Cash Value Factor per Re.1 vested Paid-up Additions for Outstanding Policy Term of 37 to 54 years\n\nSource: {'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': 'D:20230821052617', 'title': 'Exide Life Income Advantage Plan – Terms and Conditions', 'author': 'mayurc', 'moddate': 'D:20230821052617', 'source': 'Policy+Documents\\\\HDFC-Life-Sampoorna-Jeevan-101N158V04-Policy-Document (1).pdf', 'total_pages': 44, 'page': 36, 'page_label': '37'}\n Page Content: Cash Value Factor per Re.1 vested Paid-up Additions for Outstanding Policy 

In [138]:
insurance_agent("what is the life insurance coverage for disability?")

answer: The policy provides coverage for disability due to specific conditions, particularly if the individual cannot perform at least three Activities of Daily Living for a continuous period of six months. 

source: HDFC Life Group Poorna Suraksha (101N137V02) - Policy Document, page 28.
